## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import src.lme_utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xesmf as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_W = [120, 200]
LONS_E = [200, 280]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-30, 0, 30],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[50, 120 - 160],
            # xlocs=[],
            ylocs=[-30, 0, 30],
        )
        gl_.top_labels = False
        # gl_.bottom_labels = True
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_pr_diff(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance_r",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_sst_sigma(ax, data, lev_min=0.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_pr(ax, data, lev_min=1, dx=1, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.rain",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def prep(y):
    """prepare data for EOF analysis"""

    ## resample to seasonal
    y_ = y.resample({"time": "QS-FEB"}).mean()

    ## find months
    is_aso = y_.time.dt.month == 8
    is_ndj = y_.time.dt.month == 11
    is_fma = y_.time.dt.month == 2
    in_range = is_aso.values | is_ndj.values | is_fma.values

    ## subset for data
    y_ = y_.isel(time=in_range).isel(time=slice(1, -2))

    ## get year start
    yrs = y_.time.dt.year.values
    mths = y_.time.dt.month.values
    year_start = np.array([y - 1 if (m == 2) else y for (y, m) in zip(yrs, mths)])

    ## get season
    season_dict = {2: "FMA", 8: "ASO", 11: "NDJ"}
    season = np.array([season_dict[m] for m in mths])

    ## create multi-index
    new_idx = pd.MultiIndex.from_arrays([year_start, season], names=["y0", "season"])

    ## convert to xr
    new_idx_xr = xr.Coordinates.from_pandas_multiindex(new_idx, dim="time")

    ## add to original data
    y_ = y_.assign_coords(new_idx_xr)

    return y_.unstack("time")

## Data loading

In [ ]:
def sel_area(data, lon_range=None, lat_range=None):
    """select area on t grids"""

    ## find in lon range
    if lon_range is None:
        lon_idx = np.array([True for _ in data.nlon])
    else:
        lon_idx = (
            ((data.TLONG >= lon_range[0]) & (data.TLONG <= lon_range[1]))
            .any("nlat")
            .values
        )

    if lat_range is None:
        lat_idx = np.array([True for _ in data.nlat])
    else:
        lat_idx = (
            ((data.TLAT >= lat_range[0]) & (data.TLAT <= lat_range[1]))
            .any("nlon")
            .values
        )

    return data.isel(nlon=lon_idx, nlat=lat_idx)


def area_avg(data, varname, lon_range=None, lat_range=None):
    """average over area"""

    ## trim data
    data_ = sel_area(data, lon_range=lon_range, lat_range=lat_range)

    ## get dims to sum over
    integrate = lambda x: (x * data_["dA"]).sum(["nlon", "nlat"])

    return integrate(data_[varname]) / integrate(1.0)


def get_eli_helper(exceeds_thresh):
    """compute ELI from mask of threshold exceedance"""

    ## sum and count longitudes exceeding thresh
    lon = exceeds_thresh.longitude
    longitude_sum = (exceeds_thresh * lon).sum(["longitude", "latitude"])
    longitude_count = exceeds_thresh.sum(["longitude", "latitude"])

    ## eli is average longitude
    eli = longitude_sum / longitude_count

    return eli


def get_eli(rsst, rsst_thresh=0, max_lat=15):
    """compute ELI from tropical SST data"""

    ## get equatorial Pac. SST
    rsst_pac = rsst.sel(longitude=slice(120, 280), latitude=slice(-max_lat, max_lat))

    ## get boolean array where SST exceeds thresh
    exceeds_thresh0 = rsst_pac >= rsst_thresh

    ## compute initial ELI estimate
    eli0 = get_eli_helper(exceeds_thresh0)

    return eli0

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=2)
client = Client(cluster)
client

### do loading

In [ ]:
SAVE_FP = pathlib.Path("/glade/work/kcarr/lme_data/snr")
RSST_FP = SAVE_FP / "rsst_split.nc"

## load precip & RSST
pr = xr.open_dataset(SAVE_FP / "pr_split.nc").compute()
pr = pr.rename({"lon": "longitude", "lat": "latitude"})

## load SST
sst = xr.open_dataset(SAVE_FP / "sst_split.nc").compute()
sst = sst.rename({"lon": "longitude", "lat": "latitude"})

if RSST_FP.is_file():
    rsst = xr.open_dataset(RSST_FP)

else:

    ## func to compute rsst
    get_rsst = lambda x: x - x.mean(["latitude", "longitude"])

    ## get total sst
    sst["total"] = sst["forced"] + sst["anom"]

    ## compute RSST
    rsst = xr.merge([get_rsst(sst[n]).rename(n) for n in ["forced", "total"]])

    ## save to file
    rsst.to_netcdf(RSST_FP)

## update coords to match
pr = pr.assign_coords({"time": rsst.time})

## Compute indices

In [ ]:
def get_meri(x):
    """average over longitudes in monsoon region"""
    return x.sel(longitude=slice(65, 140)).mean("longitude")

In [ ]:
## precip
pr_s = get_meri(pr).sel(latitude=slice(-25, -7.5)).mean("latitude")
pr_n = get_meri(pr).sel(latitude=slice(7.5, 25)).mean("latitude")
pr_eq = get_meri(pr).sel(latitude=slice(-7.5, 7.5)).mean("latitude")

eq_avg = lambda x: x.sel(latitude=slice(-5, 5)).mean("latitude")
get_Tw = lambda x: eq_avg(x.sel(longitude=slice(120, 170)).mean("longitude"))

## temp
T34 = src.utils.get_nino34(sst)
rT34 = src.utils.get_nino34(rsst)
rT34["anom"] = rT34["total"] - rT34["forced"]

Tw = get_Tw(sst)
Tw["total"] = Tw["anom"] + Tw["forced"]

## enso longitude index
eli = get_eli(rsst)
eli["anom"] = eli["total"] - eli["forced"]

## Analysis

### ELI forced timeseries

In [ ]:
volc_forcing = src.lme_utils.load_volc_forcing(cutoff=25)

In [ ]:
resamp = lambda x: x.groupby("time.year").mean()

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")

for n, la, alpha in zip(["total", "forced"], ["ens. mean", "SNR filtered"], [0.7, 1]):

    ## get data
    if n == "total":
        data_ = resamp(eli[n]).mean("member")
    else:
        data_ = resamp(eli[n])
    # yr = data_.time.dt.year

    ## plot data
    ax.plot(data_.year, data_, label=la, alpha=alpha, lw=3)

## plot climatological value
clim_ = resamp(eli["forced"]).sel(year=slice(900, 999)).mean("year")
ax.axhline(clim_, ls="--", c="k", lw=1)


## plot volc data
ax.scatter(
    volc_forcing.year,
    clim_ * xr.ones_like(volc_forcing.year) - 5,
    s=volc_forcing["Global_norm"] * 100,
    marker="^",
    c="magenta",
    edgecolor="k",
)

ax.legend()
ax.set_title("ELI")

## formatting
ax.set_xlim([840, 2015])
ax.set_ylabel(r"ELI ($^{\circ}$E)")
src.utils.add_vticks([ax], xlines=[1258, 1762], xticks=[850, 1258, 1762, 2005])

plt.show()

Monsoon rainfall

In [ ]:
# pr_monsoon = prep(pr_n).sel(season="ASO") + prep(pr_s).sel(season="FMA")
# pr_monsoon = prep(pr_eq).sel(season=["ASO","NDJ"]).sum("season")

pr_monsoon = prep(pr_n).sum("season")+prep(pr_s).sum("season")-prep(pr_eq).sum("season")

pr_monsoon = pr_monsoon.isel(y0=slice(1,None))

In [ ]:
resamp = lambda x: x.groupby("time.year").mean()

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")

    
    ## plot data
ax.plot(pr_monsoon.y0, pr_monsoon["forced"], lw=3)

# ## plot climatological value
clim_ = pr_monsoon["forced"].sel(y0=slice(900, 999)).mean("y0")
ax.axhline(clim_, ls="--", c="k", lw=1)


## plot volc data
ax.scatter(
    volc_forcing.year,
    clim_ * xr.ones_like(volc_forcing.year) - 3,
    s=volc_forcing["Global_norm"] * 100,
    marker="^",
    c="magenta",
    edgecolor="k",
)

ax.set_title("Monsoon rainfall")

## formatting
ax.set_xlim([840, 2015])
ax.set_ylabel(r"ELI ($^{\circ}$E)")
src.utils.add_vticks([ax], xlines=[1258, 1762], xticks=[850, 1258, 1762, 2005])

plt.show()

### Scatter plots

#### ELI vs. precip.

In [ ]:
PLOT_ANOM = False

sel_aso = lambda x: prep(x).sel(season="ASO")
sel_ndj = lambda x: prep(x).sel(season="NDJ")
sel_fma = lambda x: prep(x).sel(season="FMA")

fig, axs = plt.subplots(3, 3, figsize=(10, 10), layout="constrained")
for i, sea in enumerate(["ASO", "NDJ", "FMA"]):

    ## func to select season
    sel_season = lambda x: prep(x).sel(season=sea)
    axs[0, i].set_title(sea)

    ## loop thru indices
    for j, idx in enumerate([pr_n, pr_eq, pr_s]):

        if PLOT_ANOM:

            axs[j, i].scatter(sel_season(eli["anom"]), sel_season(idx["anom"]), s=5)

        else:
            ## plot data
            axs[j, i].scatter(
                sel_season(eli["total"]), sel_season(idx["forced"] + idx["anom"]), s=5
            )

            ## guidelines
            axs[j, i].axhline(0, ls="--", c="gray", lw=1)
            for t in [210]:
                axs[j, i].axvline(t, ls="--", c="gray", lw=1)

src.utils.set_lims(axs)
for ax in axs[0, :]:
    ax.set_xticks([])
for ax in axs[:, 1:].flatten():
    ax.set_yticks([])
plt.show()

#### $T_{34}$ vs. precip

In [ ]:
PLOT_ANOM = True

## specify which index to use as x-variable
XIDX = T34

sel_aso = lambda x: prep(x).sel(season="ASO")
sel_ndj = lambda x: prep(x).sel(season="NDJ")
sel_fma = lambda x: prep(x).sel(season="FMA")

fig, axs = plt.subplots(3, 3, figsize=(10, 10), layout="constrained")
for i, sea in enumerate(["ASO", "NDJ", "FMA"]):

    ## func to select season
    sel_season = lambda x: prep(x).sel(season=sea)
    axs[0, i].set_title(sea)

    ## loop thru indices
    for j, idx in enumerate([pr_n, pr_eq, pr_s]):

        if PLOT_ANOM:

            axs[j, i].scatter(sel_season(XIDX["anom"]), sel_season(idx["anom"]), s=5)

        else:
            ## plot data
            axs[j, i].scatter(
                sel_season(XIDX["forced"] + XIDX["anom"]),
                sel_season(idx["forced"] + idx["anom"]),
                s=5,
            )

            # ## guidelines
            # axs[j,i].axhline(0, ls="--", c="gray", lw=1)
            # for t in [210]:
            #     axs[j,i].axvline(t, ls="--", c="gray", lw=1)

src.utils.set_lims(axs)
for ax in axs[0, :]:
    ax.set_xticks([])
for ax in axs[:, 1:].flatten():
    ax.set_yticks([])
plt.show()

#### ELI vs. SST

In [ ]:
PLOT_ANOM = True

## specify which index to use as x-variable
XIDX = T34

sel_aso = lambda x: prep(x).sel(season="ASO")
sel_ndj = lambda x: prep(x).sel(season="NDJ")
sel_fma = lambda x: prep(x).sel(season="FMA")

fig, axs = plt.subplots(3, 3, figsize=(10, 10), layout="constrained")
for i, sea in enumerate(["ASO", "NDJ", "FMA"]):

    ## func to select season
    sel_season = lambda x: prep(x).sel(season=sea)
    axs[0, i].set_title(sea)

    ## loop thru indices
    for j, idx in enumerate([pr_n, pr_eq, pr_s]):

        if PLOT_ANOM:

            axs[j, i].scatter(sel_season(eli["anom"]), sel_season(XIDX["anom"]), s=5)

        else:
            ## plot data
            axs[j, i].scatter(
                sel_season(eli["total"]),
                sel_season(XIDX["forced"] + XIDX["anom"]),
                s=5,
            )

            # ## guidelines
            # axs[j,i].axhline(0, ls="--", c="gray", lw=1)
            # for t in [210]:
            #     axs[j,i].axvline(t, ls="--", c="gray", lw=1)

src.utils.set_lims(axs)
for ax in axs[0, :]:
    ax.set_xticks([])
for ax in axs[:, 1:].flatten():
    ax.set_yticks([])
plt.show()

### Forced response

In [ ]:
PLOT_ANOM = False

sel_aso = lambda x: prep(x).sel(season="ASO")
sel_ndj = lambda x: prep(x).sel(season="NDJ")
sel_fma = lambda x: prep(x).sel(season="FMA")

fig, axs = plt.subplots(3, 3, figsize=(7,7), layout="constrained")
for i, sea in enumerate(["ASO", "NDJ", "FMA"]):

    ## func to select season
    sel_season = lambda x: prep(x).sel(season=sea)
    axs[0, i].set_title(sea)

    ## loop thru indices
    for j, idx in enumerate([pr_n, pr_eq, pr_s]):

        axs[j, i].scatter(sel_season(eli["forced"]), sel_season(idx["forced"]), s=5)

# src.utils.set_lims(axs)
for ax in axs[:-1, :].flatten():
    ax.set_xticks([])
for ax in axs[:, 1:].flatten():
    ax.set_yticks([])
plt.show()

### Spatial structure

In [ ]:
def get_m(x, y, dims=["time","member"]):
    """get regression coefficient"""

    ## trim
    x_ = x.sel(time=slice("0900-01","1850-12"))
    y_ = y.sel(time=slice("0900-01","1850-12"))

    ## resample
    x_ = x_.resample({"time":"YS"}).mean()
    y_ = y_.resample({"time":"YS"}).mean()

    ## subtract mean
    x_ = x_-x_.mean(dims)
    y_ = y_-y_.mean(dims)
    
    cov = (x_ * y_).mean(dims)
    var_x = (x_ * x_).mean(dims)

    return cov/var_x

In [ ]:
m_sst = get_m(x=-Tw, y=sst)
m_pr = get_m(x=-Tw, y=pr)

In [ ]:
## specify which variable to plot
# plot_data, dx = -m_sst, 0.3
plot_data, dx = m_pr, 0.5

## specify intervals
dxs = np.array([2])

fig = plt.figure(figsize=(12, 5), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=25, lon_range=(60, 280))
axs = src.utils.subplots_with_proj(fig, nrows=2, ncols=2, format_func=format_func)


for i, (plot_data, dx, la) in enumerate([[-m_sst, 0.3, "sst"], [m_pr, 0.5, "pr"]]):
    
    for j,n in enumerate(["forced","anom"]):
        cp00 = plot_pr_diff(
            axs[j, i],
            plot_data[n],
            dx=dx,
        )
    
        ## label
        axs[j, i].set_title(f"{la} ({n})", loc="left")
    
    ## add labels
    # add_gridlines(axs)
    bbox = dict(boxstyle="round", facecolor="white", alpha=1)
    legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
    for ax in axs.flatten():
        ax.set_aspect("auto")
        src.utils.plot_box(ax, lons=[120, 170], lats=[-5, 5], c="magenta", ls="--")
    
    ## colorbars
    cb_kwargs = dict(label=r"$^{\circ}$C", pad=0.03)
    # fig.colorbar(cp00, ax=axs[0,0], ticks=[-3.6,0,3.6], **cb_kwargs)
    # fig.colorbar(cp10, ax=axs[1,0], ticks=[-1.8,0,1.8], **cb_kwargs)
    add_gridlines(axs)
plt.show()